# 07 — Proprietary API Baseline (Variant E)

Run the same extraction task through strong proprietary models via **OpenRouter** to
establish an upper-bound baseline for comparison with the open-source variants (A–D).

**No GPU needed** — this notebook only makes API calls.

**Models tested:**
- `openai/GPT-5.4`
- `openai/GPT-5.4-nano`

**What we do:**

1. Load data splits, schemas, and prompt templates from previous notebooks
2. Set up the OpenRouter API client (OpenAI-compatible)
3. Run each model on a small smoke test to verify the pipeline
4. Run each model on the full val set with the same prompt as Variant A (zero-shot)
5. Parse outputs, log latency and token usage, track cost
6. Save predictions per model in the same standardized format as notebooks 05–06

In [1]:
import json
import os
import sys
import time
from pathlib import Path
from collections import Counter, defaultdict
from IPython.display import clear_output, display

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(Path("../.env"))

CLEANED_DIR = Path("../data/WDC-PAVE/cleaned")
PRED_DIR = Path("../data/WDC-PAVE/predictions")
PRED_DIR.mkdir(exist_ok=True)

# OpenRouter uses the OpenAI-compatible API
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
)

# Models to evaluate — each gets its own prediction file
MODELS = [
    "openai/GPT-5.4-nano",
    "openai/GPT-5.4",
]

def model_slug(model_name: str) -> str:
    """Convert model name to a safe filename slug."""
    return model_name.replace("/", "_").replace(":", "_")

print(f"API client ready (OpenRouter)")
print(f"Models to evaluate: {len(MODELS)}")
for m in MODELS:
    print(f"  - {m}")

API client ready (OpenRouter)
Models to evaluate: 2
  - openai/GPT-5.4-nano
  - openai/GPT-5.4


## 1. Load data and prompt templates

In [2]:
def load_jsonl(path: Path) -> list[dict]:
    with open(path) as f:
        return [json.loads(line) for line in f]

train = load_jsonl(CLEANED_DIR / "train.jsonl")
val = load_jsonl(CLEANED_DIR / "val.jsonl")
test = load_jsonl(CLEANED_DIR / "test.jsonl")

with open(CLEANED_DIR / "category_schemas.json") as f:
    CATEGORY_SCHEMAS = json.load(f)

print(f"Train: {len(train)} | Val: {len(val)} | Test: {len(test)}")

Train: 994 | Val: 142 | Test: 284


In [3]:
# Same prompt templates as notebooks 04–05
from jinja2 import Template

SYSTEM_MESSAGE = """\
You are an information extraction system. Extract product attributes from the given title and description.

Rules:
- Return exactly one valid JSON object.
- Use exactly the attribute names listed by the user as keys.
- If an attribute is missing or cannot be determined, return null.
- If multiple values apply, return them as a JSON array sorted alphabetically.
- Do not add extra keys. Do not explain. Use only information from the input text.
- The word "Null" in the input is a data artifact — ignore it."""

USER_TEMPLATE = Template("""\
Category: {{ category }}

Attributes:
{{ attribute_list }}

Title: {{ title }}

Description: {{ description }}""")


def build_messages(record: dict, schemas: dict[str, list[str]]) -> list[dict]:
    """Build chat messages for zero-shot inference."""
    category = record["category"]
    attrs = schemas[category]
    user_msg = USER_TEMPLATE.render(
        category=category,
        attribute_list=", ".join(attrs),
        title=record["input_title"],
        description=record["input_description"],
    )
    return [
        {"role": "system", "content": SYSTEM_MESSAGE},
        {"role": "user", "content": user_msg},
    ]

print("Prompt templates loaded.")

Prompt templates loaded.


## 2. API inference helper

Calls the OpenRouter API (OpenAI-compatible), parses the JSON response, and tracks
latency, token usage, and cost.

In [4]:
import re

def call_api(
    messages: list[dict],
    model: str,
    temperature: float = 0,
    max_tokens: int = 512,
    max_retries: int = 3,
) -> dict:
    """Call the OpenRouter API and return a standardized result dict.
    
    Returns:
      - raw_output: the model's text response
      - parsed_json: parsed JSON dict (or None)
      - parse_error: error message if parsing failed
      - input_tokens: prompt tokens reported by the API
      - output_tokens: completion tokens reported by the API
      - latency_s: wall-clock time for the API call
    """
    last_error = None
    for attempt in range(max_retries):
        try:
            t0 = time.perf_counter()
            response = client.chat.completions.create(
                model=model,
                messages=messages,
                temperature=temperature,
                max_tokens=max_tokens,
            )
            latency = time.perf_counter() - t0
            break
        except Exception as e:
            last_error = e
            if attempt < max_retries - 1:
                wait = 2 ** attempt
                print(f"  API error (attempt {attempt+1}): {e}. Retrying in {wait}s...")
                time.sleep(wait)
            else:
                return {
                    "raw_output": "",
                    "parsed_json": None,
                    "parse_error": f"API failed after {max_retries} retries: {last_error}",
                    "input_tokens": 0,
                    "output_tokens": 0,
                    "latency_s": 0,
                }
    
    raw_output = response.choices[0].message.content or ""
    usage = response.usage
    input_tokens = usage.prompt_tokens if usage else 0
    output_tokens = usage.completion_tokens if usage else 0
    
    # Try to parse JSON
    parsed_json = None
    parse_error = None
    try:
        parsed_json = json.loads(raw_output)
    except json.JSONDecodeError:
        # Try extracting JSON from markdown code blocks or surrounding text
        # Match ```json ... ``` or ``` ... ```
        code_block = re.search(r"```(?:json)?\s*\n?(.*?)\n?```", raw_output, re.DOTALL)
        if code_block:
            try:
                parsed_json = json.loads(code_block.group(1))
            except json.JSONDecodeError:
                pass
        
        if parsed_json is None:
            # Try bare JSON extraction
            json_match = re.search(r"\{[^{}]*(?:\{[^{}]*\}[^{}]*)*\}", raw_output, re.DOTALL)
            if json_match:
                try:
                    parsed_json = json.loads(json_match.group())
                except json.JSONDecodeError as e:
                    parse_error = f"JSON extraction failed: {e}"
            else:
                parse_error = "No JSON object found in output"
    
    return {
        "raw_output": raw_output,
        "parsed_json": parsed_json,
        "parse_error": parse_error,
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "latency_s": round(latency, 3),
    }


print("call_api() defined.")

call_api() defined.


## 3. Smoke test — one record per category

Verify the API connection works and inspect the model's outputs before running the full set.

In [5]:
smoke_records = []
for cat in CATEGORY_SCHEMAS:
    rec = next(r for r in val if r["category"] == cat)
    smoke_records.append(rec)

print(f"Smoke test: {len(smoke_records)} records (one per category)")
print(f"Testing {len(MODELS)} models\n")

smoke_results = {}  # model -> list of (rec, result)
for model_name in MODELS:
    print(f"\n{'='*60}")
    print(f"Model: {model_name}")
    print(f"{'='*60}")
    smoke_results[model_name] = []
    for rec in smoke_records:
        messages = build_messages(rec, CATEGORY_SCHEMAS)
        result = call_api(messages, model=model_name)
        smoke_results[model_name].append((rec, result))
        
        print(f"  {rec['category']} (ID={rec['id']}): "
              f"{result['latency_s']}s | {result['input_tokens']}+{result['output_tokens']} tok"
              + (f" | PARSE ERROR: {result['parse_error']}" if result["parse_error"] else " | OK"))

Smoke test: 5 records (one per category)
Testing 2 models


Model: openai/GPT-5.4-nano
  Computers And Accessories (ID=12899063): 3.048s | 361+176 tok | OK
  Home And Garden (ID=10370285): 0.907s | 216+66 tok | OK
  Office Products (ID=6361626): 9.465s | 288+94 tok | OK
  Jewelry (ID=9039991): 5.844s | 293+36 tok | OK
  Grocery And Gourmet Food (ID=15134680): 1.411s | 255+54 tok | OK

Model: openai/GPT-5.4
  Computers And Accessories (ID=12899063): 1.43s | 361+77 tok | OK
  Home And Garden (ID=10370285): 4.41s | 216+48 tok | OK
  Office Products (ID=6361626): 1.312s | 288+62 tok | OK
  Jewelry (ID=9039991): 2.257s | 293+26 tok | OK
  Grocery And Gourmet Food (ID=15134680): 1.027s | 255+40 tok | OK


In [6]:
# Compare predicted vs gold for one model's smoke test (first model)
first_model = MODELS[0]
print(f"Detailed smoke-test comparison for: {first_model}\n")

for rec, result in smoke_results[first_model]:
    print(f"=== {rec['category']} (ID={rec['id']}) ===")
    gold = rec["gold_json"]
    pred = result["parsed_json"]
    
    if pred is None:
        print("  No parsed JSON — skipping comparison")
        print(f"  Raw: {result['raw_output'][:200]}")
        print()
        continue
    
    schema = CATEGORY_SCHEMAS[rec["category"]]
    print(f"  {'Attribute':<30s} {'Gold':<40s} {'Predicted':<40s} {'Match'}")
    print(f"  {'-'*30} {'-'*40} {'-'*40} {'-'*5}")
    for attr in schema:
        g = gold.get(attr)
        p = pred.get(attr)
        g_str = json.dumps(g) if g is not None else "null"
        p_str = json.dumps(p) if p is not None else "null"
        match = "OK" if g == p else "---"
        if match == "---" and isinstance(g, str) and isinstance(p, str) and g.lower() == p.lower():
            match = "~ci"
        print(f"  {attr:<30s} {g_str:<40s} {p_str:<40s} {match}")
    print()

Detailed smoke-test comparison for: openai/GPT-5.4-nano

=== Computers And Accessories (ID=12899063) ===
  Attribute                      Gold                                     Predicted                                Match
  ------------------------------ ---------------------------------------- ---------------------------------------- -----
  Generation                     "Generation 8 Generation 9"              ["Gen8", "Gen9"]                         ---
  Part Number                    "652589S21"                              ["652589-S21", "653971-001", "641552-004", "689287-004", "666355-004", "619286-004", "507129-018"] ---
  Product Type                   "Storage Solutions"                      "Enterprise Hot-Plug Hard Drive"         ---
  Cache                          null                                     null                                     OK
  Processor Type                 null                                     null                                     OK
  

## 4. Full val set inference

In [7]:
def run_api_inference(records: list[dict], model_name: str, variant_name: str) -> list[dict]:
    """Run inference via API on a list of records using the specified model."""
    predictions = []
    total_input_tokens = 0
    total_output_tokens = 0
    n = len(records)
    
    for i, rec in enumerate(records):
        messages = build_messages(rec, CATEGORY_SCHEMAS)
        result = call_api(messages, model=model_name)
        
        total_input_tokens += result["input_tokens"]
        total_output_tokens += result["output_tokens"]
        
        predictions.append({
            "id": rec["id"],
            "category": rec["category"],
            "variant": variant_name,
            "model": model_name,
            "gold_json": rec["gold_json"],
            "pred_json": result["parsed_json"],
            "raw_output": result["raw_output"],
            "parse_error": result["parse_error"],
            "input_tokens": result["input_tokens"],
            "output_tokens": result["output_tokens"],
            "latency_s": result["latency_s"],
        })
        
        # Print progress
        pct = (i + 1) / n * 100
        print(f"\r  [{i+1}/{n}] {pct:5.1f}% — {rec['category'][:20]}", end="", flush=True)
    
    # Summary
    n_parsed = sum(1 for p in predictions if p["pred_json"] is not None)
    avg_latency = sum(p["latency_s"] for p in predictions) / len(predictions)
    print(f"\n{variant_name}: {n_parsed}/{len(predictions)} parsed "
          f"({n_parsed/len(predictions)*100:.1f}%), avg latency {avg_latency:.2f}s")
    print(f"Total tokens: {total_input_tokens:,} input + {total_output_tokens:,} output "
          f"= {total_input_tokens + total_output_tokens:,}")
    
    return predictions


def save_predictions(predictions: list[dict], path: Path):
    with open(path, "w") as f:
        for pred in predictions:
            f.write(json.dumps(pred) + "\n")
    print(f"Saved {len(predictions)} predictions to {path}")


print("run_api_inference() defined.")

run_api_inference() defined.


In [8]:
all_preds = {}  # model_name -> predictions list

for model_name in MODELS:
    slug = model_slug(model_name)
    variant_name = f"E_{slug}"
    out_path = PRED_DIR / f"variant_E_{slug}_val.jsonl"
    
    print(f"\n{'='*60}")
    print(f"Running: {model_name}")
    print(f"Output:  {out_path.name}")
    print(f"{'='*60}")
    
    preds = run_api_inference(val, model_name=model_name, variant_name=variant_name)
    save_predictions(preds, out_path)
    all_preds[model_name] = preds


Running: openai/GPT-5.4-nano
Output:  variant_E_openai_GPT-5.4-nano_val.jsonl
  [142/142] 100.0% — Home And Gardenccess
E_openai_GPT-5.4-nano: 142/142 parsed (100.0%), avg latency 1.62s
Total tokens: 36,042 input + 11,395 output = 47,437
Saved 142 predictions to ../data/WDC-PAVE/predictions/variant_E_openai_GPT-5.4-nano_val.jsonl

Running: openai/GPT-5.4
Output:  variant_E_openai_GPT-5.4_val.jsonl
  [142/142] 100.0% — Home And Gardenccess
E_openai_GPT-5.4: 142/142 parsed (100.0%), avg latency 1.49s
Total tokens: 36,042 input + 7,824 output = 43,866
Saved 142 predictions to ../data/WDC-PAVE/predictions/variant_E_openai_GPT-5.4_val.jsonl


## 5. Quick metrics

In [9]:
for model_name, preds in all_preds.items():
    n = len(preds)
    n_parsed = sum(1 for p in preds if p["pred_json"] is not None)
    n_schema_ok = 0
    n_exact = 0

    for p in preds:
        if p["pred_json"] is not None:
            expected_keys = set(CATEGORY_SCHEMAS[p["category"]])
            actual_keys = set(p["pred_json"].keys())
            if expected_keys == actual_keys:
                n_schema_ok += 1
            if p["pred_json"] == p["gold_json"]:
                n_exact += 1

    latencies = [p["latency_s"] for p in preds]
    input_toks = [p["input_tokens"] for p in preds]
    output_toks = [p["output_tokens"] for p in preds]

    print(f"\nVariant E — {model_name}")
    print(f"{'='*50}")
    print(f"  Records:          {n}")
    print(f"  Valid JSON:       {n_parsed}/{n} ({n_parsed/n*100:.1f}%)")
    print(f"  Schema match:     {n_schema_ok}/{n} ({n_schema_ok/n*100:.1f}%)")
    print(f"  Exact match:      {n_exact}/{n} ({n_exact/n*100:.1f}%)")
    print(f"  Avg latency:      {sum(latencies)/n:.2f}s")
    print(f"  Avg input tokens: {sum(input_toks)/n:.0f}")
    print(f"  Avg output tokens:{sum(output_toks)/n:.0f}")
    print(f"  Total tokens:     {sum(input_toks) + sum(output_toks):,}")


Variant E — openai/GPT-5.4-nano
  Records:          142
  Valid JSON:       142/142 (100.0%)
  Schema match:     141/142 (99.3%)
  Exact match:      0/142 (0.0%)
  Avg latency:      1.62s
  Avg input tokens: 254
  Avg output tokens:80
  Total tokens:     47,437

Variant E — openai/GPT-5.4
  Records:          142
  Valid JSON:       142/142 (100.0%)
  Schema match:     142/142 (100.0%)
  Exact match:      0/142 (0.0%)
  Avg latency:      1.49s
  Avg input tokens: 254
  Avg output tokens:55
  Total tokens:     43,866


In [10]:
# Per-category breakdown for each model
for model_name, preds in all_preds.items():
    print(f"\nPer-category breakdown: {model_name}")
    print(f"  {'Category':<35s} {'Valid JSON':>10s} {'Schema OK':>10s} {'Exact':>10s} {'n':>4s}")
    print(f"  {'-'*35} {'-'*10} {'-'*10} {'-'*10} {'-'*4}")

    cat_stats = defaultdict(lambda: {"total": 0, "parsed": 0, "schema_ok": 0, "exact": 0})
    for p in preds:
        cat_stats[p["category"]]["total"] += 1
        if p["pred_json"] is not None:
            cat_stats[p["category"]]["parsed"] += 1
            expected = set(CATEGORY_SCHEMAS[p["category"]])
            if set(p["pred_json"].keys()) == expected:
                cat_stats[p["category"]]["schema_ok"] += 1
            if p["pred_json"] == p["gold_json"]:
                cat_stats[p["category"]]["exact"] += 1

    for cat in CATEGORY_SCHEMAS:
        s = cat_stats[cat]
        t = s["total"]
        print(f"  {cat:<35s} {s['parsed']/t*100:>9.1f}% {s['schema_ok']/t*100:>9.1f}% "
              f"{s['exact']/t*100:>9.1f}% {t:>4d}")


Per-category breakdown: openai/GPT-5.4-nano
  Category                            Valid JSON  Schema OK      Exact    n
  ----------------------------------- ---------- ---------- ---------- ----
  Computers And Accessories               100.0%     100.0%       0.0%   44
  Home And Garden                         100.0%     100.0%       0.0%   35
  Office Products                         100.0%      96.7%       0.0%   30
  Jewelry                                 100.0%     100.0%       0.0%   25
  Grocery And Gourmet Food                100.0%     100.0%       0.0%    8

Per-category breakdown: openai/GPT-5.4
  Category                            Valid JSON  Schema OK      Exact    n
  ----------------------------------- ---------- ---------- ---------- ----
  Computers And Accessories               100.0%     100.0%       0.0%   44
  Home And Garden                         100.0%     100.0%       0.0%   35
  Office Products                         100.0%     100.0%       0.0%   30
  J

## 6. Inspect failures (if any)

In [11]:
for model_name, preds in all_preds.items():
    print(f"\n{'='*60}")
    print(f"Failures for: {model_name}")
    print(f"{'='*60}")
    
    # Parse failures
    parse_failures = [p for p in preds if p["pred_json"] is None]
    print(f"Parse failures: {len(parse_failures)} / {len(preds)}")
    for p in parse_failures[:3]:
        print(f"  ID={p['id']} ({p['category']})")
        print(f"  Error: {p['parse_error']}")
        print(f"  Raw: {p['raw_output'][:200]}")
        print()

    # Schema mismatches
    schema_mismatches = []
    for p in preds:
        if p["pred_json"] is not None:
            expected = set(CATEGORY_SCHEMAS[p["category"]])
            actual = set(p["pred_json"].keys())
            if expected != actual:
                schema_mismatches.append(p)

    print(f"Schema mismatches: {len(schema_mismatches)} / {len(preds)}")
    for p in schema_mismatches[:3]:
        expected = set(CATEGORY_SCHEMAS[p["category"]])
        actual = set(p["pred_json"].keys())
        print(f"  ID={p['id']} ({p['category']})")
        if expected - actual:
            print(f"    Missing: {expected - actual}")
        if actual - expected:
            print(f"    Extra:   {actual - expected}")


Failures for: openai/GPT-5.4-nano
Parse failures: 0 / 142
Schema mismatches: 1 / 142
  ID=5061601 (Office Products)
    Missing: {'Color(s)'}
    Extra:   {'Color(s'}

Failures for: openai/GPT-5.4
Parse failures: 0 / 142
Schema mismatches: 0 / 142


## 7. List all prediction files

Show all prediction files across all notebooks — this is the input to notebook 08 (evaluation).

In [12]:
print("All prediction files:")
for p in sorted(PRED_DIR.iterdir()):
    if p.suffix == ".jsonl":
        n_lines = sum(1 for _ in open(p))
        size_kb = p.stat().st_size / 1024
        print(f"  {p.name:50s} {n_lines:>4} records  {size_kb:>8.1f} KB")

All prediction files:
  variant_E_openai_GPT-5.4-nano_val.jsonl             142 records     131.3 KB
  variant_E_openai_GPT-5.4_val.jsonl                  142 records     121.3 KB


## Summary

**Models tested via OpenRouter:**
- `openai/GPT-5.4`
- `openai/GPT-5.4-nano`

**Purpose:** Establish proprietary upper-bound baselines. The same zero-shot prompt as
Variant A is sent to each model — this tells us how much of the gap between
open-source variants and perfect performance is due to model capability vs prompt quality.

**Output files (one per model):**
- `data/WDC-PAVE/predictions/variant_E_openai_GPT-5.4_val.jsonl`
- `data/WDC-PAVE/predictions/variant_E_openai_GPT-5.4-nano_val.jsonl`

**Next steps:**
- Notebook 08: full evaluation across all variants (A, B, C, D, E) with field-level F1
- Notebook 09: results comparison and visualization